# SigAlg's `L2.inner` method

In [ ]:
# If running in Google Colab, uncomment the line below and run this cell first.
# Also, for Mac+Chrome users, beware of a known bug with LaTeX redering in Colab: https://github.com/googlecolab/colabtools/issues/3192

# !pip install sigalg

The `L2.inner` method in SigAlg computes the *inner product* of two random variables in the $L^2$-Hilbert space. The API reference is [here](https://johnmyers-phd.com/sigalg/api/modules/l2/#sigalg.l2.L2.inner).

## Mathematical definition

Let $X, Y \in L^2(\Omega, \mathcal{F}, P)$ be random variables on a probability space $(\Omega, \mathcal{F}, P)$. The *inner product* of $X$ and $Y$ is defined as

$$
\langle X, Y \rangle \stackrel{\text{def}}{=} \int_\Omega XY \, dP = E(XY).
$$

This inner product makes $L^2(\Omega, \mathcal{F}, P)$ into a *Hilbert space*. It satisfies the following properties:

1. *Bilinearity*: For all $X, Y, Z \in L^2$ and $a, b \in \mathbb{R}$,
   $$
   \langle aX + bY, Z \rangle = a\langle X, Z \rangle + b\langle Y, Z \rangle,
   $$
   $$
   \langle X, aY + bZ \rangle = a\langle X, Y \rangle + b\langle X, Z \rangle.
   $$

2. *Symmetry*: For all $X, Y \in L^2$,
   $$
   \langle X, Y \rangle = \langle Y, X \rangle.
   $$

3. *Positive-definiteness*: For all $X \in L^2$,
   $$
   \langle X, X \rangle \geq 0,
   $$
   with equality if and only if $X = 0$ almost surely.

The inner product also satisfies the *Cauchy-Schwarz inequality*:

$$
|\langle X, Y \rangle| \leq \|X\| \cdot \|Y\|,
$$

where $\|X\| = \sqrt{\langle X, X \rangle}$ is the $L^2$-norm.

## API examples


### Basic inner products

We begin by setting up a probability space and creating an $L^2$ space.

In [ ]:
from sigalg.core import ProbabilityMeasure, RandomVariable, SampleSpace, SigmaAlgebra
from sigalg.l2 import L2

Omega = SampleSpace().from_sequence(size=4)

F = SigmaAlgebra(sample_space=Omega, name="F").from_dict(
    {
        0: 0,
        1: 1,
        2: 0,
        3: 1,
    }
)

P = ProbabilityMeasure(sample_space=Omega).from_dict(
    {
        0: 0.1,
        1: 0.15,
        2: 0.45,
        3: 0.3,
    }
)

H = L2(sample_space=Omega, sigma_algebra=F, probability_measure=P)

Create two random variables in the $L^2$ space.

In [ ]:
X = RandomVariable(domain=Omega, name="X").from_dict(
    {
        0: 2,
        1: -1,
        2: 2,
        3: -1,
    }
)

Y = RandomVariable(domain=Omega, name="Y").from_dict(
    {
        0: 3,
        1: 5,
        2: 3,
        3: 5,
    }
)

print(f"X: {X}")
print(f"Y: {Y}")

Compute the inner product $\langle X, Y \rangle = E(XY)$.

In [ ]:
inner_XY = H.inner(X, Y)
print(f"⟨X, Y⟩ = {inner_XY}")

The inner product $\langle X, X \rangle$ gives the squared $L^2$ norm.

In [ ]:
inner_XX = H.inner(X, X)
print(f"⟨X, X⟩ = {inner_XX}")
print(f"E(X²) = {inner_XX}")

### Symmetry

The inner product is symmetric: $\langle X, Y \rangle = \langle Y, X \rangle$.

In [ ]:
inner_YX = H.inner(Y, X)
print(f"⟨X, Y⟩ = {inner_XY}")
print(f"⟨Y, X⟩ = {inner_YX}")
print(f"Symmetric: {abs(inner_XY - inner_YX) < 1e-10}")

### Bilinearity

The inner product is bilinear. Let's verify $\langle aX + bY, Z \rangle = a\langle X, Z \rangle + b\langle Y, Z \rangle$.

In [ ]:
Z = RandomVariable(domain=Omega, name="Z").from_dict(
    {
        0: 1,
        1: -2,
        2: 1,
        3: -2,
    }
)

a, b = 2.5, -1.3

# Left side: ⟨aX + bY, Z⟩
left = H.inner(a * X + b * Y, Z)

# Right side: a⟨X, Z⟩ + b⟨Y, Z⟩
right = a * H.inner(X, Z) + b * H.inner(Y, Z)

print(f"⟨{a}X + {b}Y, Z⟩ = {left:.6f}")
print(f"{a}⟨X, Z⟩ + {b}⟨Y, Z⟩ = {right:.6f}")
print(f"Bilinearity holds: {abs(left - right) < 1e-10}")

### Cauchy-Schwarz inequality

The Cauchy-Schwarz inequality states that $|\langle X, Y \rangle| \leq \|X\| \cdot \|Y\|$. Let's verify this.

In [ ]:
import numpy as np

inner_XY = H.inner(X, Y)
norm_X = H.norm(X)
norm_Y = H.norm(Y)

print(f"|⟨X, Y⟩| = {abs(inner_XY):.6f}")
print(f"‖X‖ · ‖Y‖ = {norm_X * norm_Y:.6f}")
print(f"Cauchy-Schwarz holds: {abs(inner_XY) <= norm_X * norm_Y + 1e-10}")

### Orthogonality

Two random variables are *orthogonal* if their inner product is zero. The orthonormal basis vectors of $L^2$ are pairwise orthogonal.

In [ ]:
basis = H.basis
print("Orthogonality of basis vectors:")
for i, phi_i in basis.items():
    for j, phi_j in basis.items():
        if i != j:
            inner_prod = H.inner(phi_i, phi_j)
            print(f"  ⟨φ_{i}, φ_{j}⟩ = {inner_prod:.10f}")

### Connection to covariance

The *covariance* of two random variables can be expressed using the inner product:

$$
\text{Cov}(X, Y) = \langle X - E(X), Y - E(Y) \rangle = E\left[(X - E(X))(Y - E(Y))\right].
$$

Let's compute the covariance using both the `cov` method and the inner product.

In [ ]:
from sigalg.core import Operators

E = Operators.expectation

# Using the cov method
X.probability_measure = P
Y.probability_measure = P
cov_XY = Operators.cov(X, Y).item()

# Using the inner product
X_centered = X - E(X)
Y_centered = Y - E(Y)
cov_XY_inner = H.inner(X_centered, Y_centered)

print(f"Cov(X, Y) using cov method: {cov_XY:.6f}")
print(f"Cov(X, Y) using inner product: {cov_XY_inner:.6f}")
print(f"Match: {abs(cov_XY - cov_XY_inner) < 1e-10}")